# Example use case of PynPoint-IFS

# Import statements and pipeline initialisation

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '5'
from astropy.io import fits
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pathlib import Path
from glob import glob
import numpy as np
import sys
import matplotlib as mpl
from photutils.centroids import centroid_quadratic
from time import time
from scipy import interpolate
from scipy import ndimage,signal
from PyAstronomy.pyasl import crosscorrRV
from scipy.interpolate import UnivariateSpline
from skimage.registration import phase_cross_correlation

In [ ]:
from spyffier import Pipeline
from spyffier.utils import calibrate_wavelength_frame,plot_data,plot_spectrum,diagnostic_wvl_solution,get_sky_calc_model

In [ ]:
# path where esorex is installed
esorex_path = '/home/ipa/quanz/user_accounts/jhayoz/ESO_2024-07-22/bin/esorex'
# path to the root of where the data created by the pipeline should be saved
reduction_path = '/home/ipa/quanz/user_accounts/jhayoz/Projects/SpyFFIER/example_pipeline/'
# path to where the raw data is
raw_path = '/home/ipa/quanz/shared/eris/P112_atmospheres/HR8799/2023_10_14/raw/'

In [ ]:
pipeline = Pipeline(esorex_path = esorex_path, reduction_path = reduction_path, raw_path = raw_path, wavel_setting = 'K_long')

# Execute the reduction with the standard pipeline

In [ ]:
pipeline.do_standard_reduction(
    spiffier_gw ='K_long',
    spiffier_psw = '25mas'
)

# Execute science_ifu_jitter a first time

In [ ]:
unique_dit_ndit = pipeline._identify_science_dit_ndit()

In [ ]:
dit,ndit = unique_dit_ndit[1]
ndit=int(ndit)

In [ ]:
# identify the SKY-OBJ pair groups
# here only one group, as there are no SKY frames in this observation
files_groups = pipeline._identify_obj_sky_groups_by_time(obj_tag = 'OBJECT',sky_tag = 'SKY', dit = dit, ndit = ndit,spiffier_gw = 'K_long', spiffier_psw = '25mas')

In [ ]:
# overwrite the configuration
# Any argument that is used by the science_ifu_jitter can be changed here
new_config = {'cube.combine':'FALSE','aj-method':0,'sky_tweak':'0','skip_sky_oh_align':'TRUE','tbsub':'TRUE','skip_oh_align':'TRUE'}

In [ ]:
for files_group_i in files_groups.keys():
    obj_files = [file for file_i,file in enumerate(files_groups[files_group_i]['OBJECT'])]
    sky_files = [file for file_i,file in enumerate(files_groups[files_group_i]['SKY'])]
    pipeline.science_ifu_jitter(dit = dit, ndit = ndit,spiffier_gw = 'K_long', spiffier_psw = '25mas', obj_date_obs = obj_files, sky_date_obs = sky_files,new_config=new_config, output_name = f'product_g_{files_group_i}', use_corr_wavemap = None)

# Correct the wavelength calibration

In [ ]:
for files_group_i in list(files_groups.keys()):
    pipeline.calib_wavelength_xcorr_full(
            input_folder=f'product_g_{files_group_i}',
            output_folder=f'wvlcorr_product_g_{files_group_i}',
            continuum_sigma=20,
            accuracy=100,
            method_tellurics='transmission', # transmission, emission
            method_slit='parabola', # raw, corr, median, linear, spline, fov-linear
            method_high_order='spline', # 0-order, spline
            spline_order=2,
            spline_smoothing=0.4,
            window_size=200,
            window_shift_ratio=4,
            plot=True,save_result=True
        )

In [ ]:
# identify bad frames where the wvl calibration failed
bad_files = {1:[5]}

In [ ]:
for files_group_i in list(files_groups.keys()):
    obj_files = [file for file_i,file in enumerate(files_groups[files_group_i]['OBJECT'])]
    for file_i,files in enumerate(obj_files):
        print(file_i,files)
for files_group_i in list(files_groups.keys()):
    bad_files_names = []
    if files_group_i in bad_files.keys():
        bad_files_names = bad_files[files_group_i]
    obj_files = [file for file_i,file in enumerate(files_groups[files_group_i]['OBJECT']) if not file_i in bad_files_names]
    print(files_group_i)
    for file_i,files in enumerate(obj_files):
        print(file_i,files)

In [ ]:
# re-do science_ifu_jitter, but this time with the updated wavelength calibration (use_corr_wavemap = f'wvlcorr_product_g_{files_group_i}')
# Since the corrected wavelength maps and the science products are saved in different folders, we can use the same name to remember which corrected wavemap was used to produce which science products
# here remove the bad files identified above

In [ ]:
for files_group_i in list(files_groups.keys()):
    print(files_group_i)
    bad_files_names = []
    if files_group_i in bad_files.keys():
        bad_files_names = bad_files[files_group_i]
    obj_files = [file for file_i,file in enumerate(files_groups[files_group_i]['OBJECT']) if not file_i in bad_files_names]
    sky_files = [file for file_i,file in enumerate(files_groups[files_group_i]['SKY'])]
    for dateobs in obj_files:
        pipeline.science_ifu_jitter(dit = dit, ndit = ndit,spiffier_gw = 'K_long', spiffier_psw = '25mas', obj_date_obs = [dateobs], sky_date_obs = sky_files, new_config = new_config,output_name = f'wvlcorr_product_g_{files_group_i}', use_corr_wavemap = f'wvlcorr_product_g_{files_group_i}')

## Check wavelength solution

In [ ]:
# The wavelength error should be smaller than it was before calibration, and should look as flat as possible.

In [ ]:
for files_group_i in list(files_groups.keys()):
    pipeline.calib_wavelength_xcorr_full(
            input_folder=f'wvlcorr_product_g_{files_group_i}',
            output_folder=f'wvlcorr_checkproduct_g_{files_group_i}',
            continuum_sigma=20,
            accuracy=100,
            method_tellurics='transmission', # transmission, emission
            method_slit='median', # raw, corr, median, linear, spline, fov-linear
            method_high_order='0-order', # 0-order, spline
            spline_order=2,
            spline_smoothing=0.4,
            window_size=200,
            window_shift_ratio=4,
            plot=True,save_result=False
        )